# Workshop 10 — Random Weight Initialization

This notebook studies how initialization changes signal propagation and convergence in a deep MLP.

## Questions
- What happens when random weights are too large or poorly scaled?
- Why do exploding and vanishing activations/gradients occur?
- Why is He/Kaiming commonly paired with ReLU?
- Why is Xavier/Glorot commonly paired with tanh/sigmoid-style networks?
- How much can initialization change training time?

> **Implementation note:** the saved workshop material contains a 220-input / 20-output display example, while the recorded training experiment uses 100 one-hot positions for A, 100 for B, and 19 for C: **219 inputs and 19 output classes**. This notebook uses the actual training configuration.

## 1. Initialization formulas

### He / Kaiming normal
For ReLU-like activations:

[
\sigma = \sqrt{\frac{2}{fan_{in}}}
]

### Xavier / Glorot normal
For variance balancing across input/output dimensions:

[
\sigma = \sqrt{\frac{2}{fan_{in}+fan_{out}}}
]

The goal is not merely "random weights"; it is a random scale that keeps useful signal and gradients alive through depth.

In [ ]:
# 2. Setup
import itertools
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt

SEED = 42

def reset_random():
    torch.manual_seed(SEED)
    np.random.seed(SEED)
    random.seed(SEED)

reset_random()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

In [ ]:
# 3. Uniform vs normal distributions
reset_random()

uniform = np.random.uniform(-1, 1, 10_000)
normal = np.random.normal(0, 1, 10_000)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(uniform, bins=40)
axes[0].set_title("Uniform(-1, 1)")
axes[0].set_xlabel("value")

axes[1].hist(normal, bins=40)
axes[1].set_title("Normal(mean=0, std=1)")
axes[1].set_xlabel("value")

plt.tight_layout()
plt.show()

print("Uniform mean/std:", uniform.mean(), uniform.std())
print("Normal mean/std:", normal.mean(), normal.std())

In [ ]:
# 4. Dataset: (A * B) % (C + 1)

def one_hot(n, num_classes):
    v = np.zeros(num_classes, dtype=np.float32)
    v[n] = 1.0
    return v

pairs = list(itertools.product(range(100), range(100), range(19)))
random.shuffle(pairs)

X_all, y_all = [], []

for a, b, c in pairs:
    x = np.concatenate([
        one_hot(a, 100),
        one_hot(b, 100),
        one_hot(c, 19),
    ])
    y = (a * b) % (c + 1)
    X_all.append(x)
    y_all.append(y)

X_all = np.asarray(X_all, dtype=np.float32)
y_all = np.asarray(y_all, dtype=np.int64)

split = int(0.8 * len(X_all))
X_train, X_test = X_all[:split], X_all[split:]
y_train, y_test = y_all[:split], y_all[split:]

BATCH_SIZE = 256

train_ds = TensorDataset(
    torch.tensor(X_train),
    torch.tensor(y_train, dtype=torch.long),
)
test_ds = TensorDataset(
    torch.tensor(X_test),
    torch.tensor(y_test, dtype=torch.long),
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE)

print(f"Total: {len(X_all):,}")
print(f"Train: {len(train_ds):,}")
print(f"Test: {len(test_ds):,}")
print("Input dim:", X_train.shape[1])
print("Output classes:", len(np.unique(y_all)))

In [ ]:
# 5. Deep MLP

class DNN(nn.Module):
    def __init__(self, input_dim=219, output_dim=19, use_tanh=False):
        super().__init__()
        act = nn.Tanh if use_tanh else nn.ReLU

        self.net = nn.Sequential(
            nn.Linear(input_dim, 512), act(),
            nn.Linear(512, 512), act(),
            nn.Linear(512, 256), act(),
            nn.Linear(256, 256), act(),
            nn.Linear(256, 128), act(),
            nn.Linear(128, output_dim),
        )

    def forward(self, x):
        return self.net(x)

model = DNN()
print(model)
print("Trainable parameters:", sum(p.numel() for p in model.parameters()))

In [ ]:
# 6. Initialization + training utilities

LR = 1e-3
WEIGHT_DECAY = 0.1
MAX_EPOCHS = 100

all_runs = []

def init_weights(module, method):
    if isinstance(module, nn.Linear):
        if method == "uniform":
            nn.init.uniform_(module.weight, -1, 1)
        elif method == "he":
            nn.init.kaiming_normal_(module.weight, nonlinearity="relu")
        elif method == "xavier":
            nn.init.xavier_normal_(module.weight)

        if module.bias is not None:
            nn.init.zeros_(module.bias)

def evaluate(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            total_loss += criterion(logits, yb).item() * len(yb)
            correct += (logits.argmax(dim=1) == yb).sum().item()
            total += len(yb)

    return total_loss / total, correct / total

def train_run(init_method, use_tanh, run_name):
    reset_random()

    model = DNN(use_tanh=use_tanh).to(device)
    model.apply(lambda m: init_weights(m, init_method))

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY,
    )
    criterion = nn.CrossEntropyLoss()

    history = {
        "epoch": [],
        "train_loss": [],
        "test_loss": [],
        "train_acc": [],
        "test_acc": [],
    }

    converged_epoch = None

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()

        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)

            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

        train_loss, train_acc = evaluate(model, train_loader, criterion)
        test_loss, test_acc = evaluate(model, test_loader, criterion)

        history["epoch"].append(epoch)
        history["train_loss"].append(train_loss)
        history["test_loss"].append(test_loss)
        history["train_acc"].append(train_acc)
        history["test_acc"].append(test_acc)

        print(
            f"{run_name:14s} | epoch {epoch:3d} | "
            f"train {train_acc:6.2%} | test {test_acc:6.2%}"
        )

        if train_acc == 1.0 and test_acc == 1.0:
            converged_epoch = epoch
            break

    all_runs.append({
        "name": run_name,
        "converged_epoch": converged_epoch,
        "history": history,
    })

    return model, history

## 7. Run the four workshop comparisons

The recorded workshop notebook produced:
- He + ReLU → 100% at epoch 25
- Xavier + ReLU → 100% at epoch 49
- Uniform + ReLU → 100% at epoch 85
- Xavier + Tanh → 98.3% test accuracy at epoch 100, not strict 100%

Re-running can produce small differences depending on environment, but the comparison structure stays the same.

In [ ]:
uniform_relu_model, _ = train_run(
    init_method="uniform",
    use_tanh=False,
    run_name="Uniform+ReLU",
)

In [ ]:
he_relu_model, _ = train_run(
    init_method="he",
    use_tanh=False,
    run_name="He+ReLU",
)

In [ ]:
xavier_tanh_model, _ = train_run(
    init_method="xavier",
    use_tanh=True,
    run_name="Xavier+Tanh",
)

In [ ]:
xavier_relu_model, _ = train_run(
    init_method="xavier",
    use_tanh=False,
    run_name="Xavier+ReLU",
)

In [ ]:
# 8. Compare convergence curves

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for run in all_runs:
    h = run["history"]

    axes[0].plot(
        h["epoch"],
        h["test_acc"],
        label=run["name"],
        linewidth=2,
    )

    axes[1].plot(
        h["epoch"],
        h["test_loss"],
        label=run["name"],
        linewidth=2,
    )

axes[0].set_title("Test Accuracy by Initialization")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].set_title("Test Loss by Initialization")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].set_yscale("log")
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()

print("\nConvergence summary")
print("-" * 45)
for run in all_runs:
    print(
        f"{run['name']:16s}: "
        f"{run['converged_epoch'] if run['converged_epoch'] else 'not within 100 epochs'}"
    )

## 9. Extra diagnostic: activation and gradient scale

Training curves show *what* happened. This diagnostic helps inspect *why*.

For one batch, we record:
- standard deviation of each Linear layer output
- standard deviation of each weight gradient after one backward pass

If these statistics shrink dramatically with depth, the network may be heading toward vanishing behavior. If they grow dramatically, it may be heading toward exploding behavior.

In [ ]:
def measure_signal_and_gradient(init_method, use_tanh=False):
    reset_random()

    model = DNN(use_tanh=use_tanh).to(device)
    model.apply(lambda m: init_weights(m, init_method))

    xb, yb = next(iter(train_loader))
    xb, yb = xb.to(device), yb.to(device)

    activation_stats = []
    hooks = []

    def make_hook(name):
        def hook(module, inputs, output):
            activation_stats.append(
                (name, output.detach().std().item())
            )
        return hook

    for name, module in model.named_modules():
        if isinstance(module, nn.Linear):
            hooks.append(
                module.register_forward_hook(make_hook(name))
            )

    logits = model(xb)
    loss = F.cross_entropy(logits, yb)
    loss.backward()

    gradient_stats = [
        (name, p.grad.detach().std().item())
        for name, p in model.named_parameters()
        if "weight" in name and p.grad is not None
    ]

    for h in hooks:
        h.remove()

    return activation_stats, gradient_stats

diagnostics = [
    ("Uniform+ReLU", "uniform", False),
    ("He+ReLU", "he", False),
    ("Xavier+ReLU", "xavier", False),
    ("Xavier+Tanh", "xavier", True),
]

for name, init_method, use_tanh in diagnostics:
    acts, grads = measure_signal_and_gradient(
        init_method,
        use_tanh,
    )

    print(f"\n{name}")
    print("Activation std:")
    for layer, value in acts:
        print(f"  {layer:10s} {value:.6f}")

    print("Gradient std:")
    for layer, value in grads:
        print(f"  {layer:18s} {value:.6f}")

## Takeaways

1. Random initialization is necessary for symmetry breaking, but arbitrary scale is dangerous.
2. Repeated matrix multiplications can amplify or suppress signal and gradients.
3. Initialization should be chosen together with the activation function.
4. He/Kaiming is designed for ReLU-like activations.
5. Xavier/Glorot balances fan-in and fan-out and is commonly associated with tanh/sigmoid-style networks.
6. In the saved workshop run, He + ReLU converged fastest.
7. A poor initialization can mean many more epochs and therefore more GPU time and cost.
8. Record actual experimental outcomes separately from simplified slide-level expectations.